# E-Commerce Customer Churn & Behaviour — Narrative Walkthrough

This notebook is a readable companion to the scripts in `src/`. It runs the
same PySpark pipeline interactively so you can show the analysis step by step.

**Research questions**
1. What proportion of customers churn, and how do churned vs retained differ?
2. Which RFM segments are most at risk?
3. Which model predicts churn best, and which features matter most?

> Run from the project root so the `src` package imports correctly.
> If `data/raw/ecommerce_churn.csv` is missing, generate the synthetic sample
> in the next cell.

In [ ]:
import os, sys
# Make the project root importable when running from notebooks/.
sys.path.insert(0, os.path.abspath('..'))

from src import config
from src.generate_sample_data import generate

if not os.path.exists(config.RAW_CSV):
    os.makedirs(os.path.dirname(config.RAW_CSV), exist_ok=True)
    generate(rows=10000).to_csv(config.RAW_CSV, index=False)
    print('Generated synthetic sample at', config.RAW_CSV)
else:
    print('Using existing dataset at', config.RAW_CSV)

## 1. Start Spark and load the raw data

In [ ]:
from src.utils import get_spark, load_raw, null_report

spark = get_spark('notebook')
raw = load_raw(spark)
print('rows:', raw.count(), '| cols:', len(raw.columns))
raw.printSchema()
raw.show(5)

## 2. Clean (Objective 2): de-dup, normalise categories, impute medians

In [ ]:
from src.pipeline_01_data_cleaning import clean

print('Nulls before:')
null_report(raw).show(vertical=True, truncate=False)

cleaned = clean(raw)
print('Nulls after:')
null_report(cleaned).show(vertical=True, truncate=False)
cleaned.groupBy(config.TARGET).count().orderBy(config.TARGET).show()

## 3. EDA (Q1): churn rate and churned-vs-retained differences

In [ ]:
from pyspark.sql import functions as F

total = cleaned.count()
churn_rate = cleaned.agg(F.avg(config.TARGET)).first()[0]
print(f'Overall churn rate: {churn_rate:.2%} (n={total})')

num_cols = [c for c in config.NUMERIC_COLS if c in cleaned.columns]
cleaned.groupBy(config.TARGET).agg(
    *[F.round(F.avg(c), 2).alias(c) for c in num_cols]
).orderBy(config.TARGET).show(truncate=False)

## 4. RFM segmentation (Q2)

In [ ]:
from src.pipeline_03_rfm import add_rfm_scores, segment

rfm = segment(add_rfm_scores(cleaned))
rfm.groupBy('Segment').agg(
    F.count('*').alias('customers'),
    F.round(F.avg(config.TARGET) * 100, 2).alias('churn_rate_pct'),
).orderBy(F.desc('churn_rate_pct')).show()

## 5. Churn modelling (Q3): compare classifiers and inspect feature importances

This calls the full modelling stage, which trains Logistic Regression, Random
Forest and GBT, prints metrics (precision / recall / F1 / ROC-AUC) and saves
figures + tables to `outputs/`.

In [ ]:
spark.stop()  # the stage manages its own SparkSession
from src import pipeline_04_churn_model as model_stage
model_stage.main()

In [ ]:
import json
with open(os.path.join(config.OUTPUTS_DIR, 'model_metrics.json')) as fh:
    print(json.dumps(json.load(fh), indent=2))

## 6. Conclusions

- See `docs/recommendations.md` for retention actions (Objective 6).
- See `docs/limitations_and_future_work.md` for limitations (Objective 7).
- Figures are in `outputs/figures/`, tables in `outputs/tables/`.